# 平面三连杆：操作空间控制（OSC）实验室

本 Notebook 是一份**边实现、边验证**的练习讲义。你已经有运动学、RNEA、正向动力学和关节控制器；这次只控制末端在世界坐标系中的二维位置。四个 OSC 核心函数仍在 `src/three_link_robot/` 中由你填写，Notebook 不会替它们给出可用于仿真的答案。

完成后应能回答三个问题：① 末端加速度怎样由关节加速度得到？② 任务空间加速度怎样变成关节力矩？③ 三个关节控制二维位置时，多出的一维自由度怎样使用？

**建议顺序**：先运行环境、构型与参考轨迹单元格；完成 OSC-01、02、03 后运行定点和圆轨迹；最后完成 OSC-04 并尝试零空间姿态实验。未完成练习时会显示提示，绝不把占位结果当成仿真结果。完整推导和验收要求见 `docs/osc_learning.md`。

## 1. 准备环境

在工程根目录的终端执行：

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev,notebook]'
python -m jupyterlab
```

选择这个 `.venv` 对应的 Python 内核。若 `.venv` 从旧项目复制过来，重新执行安装命令；Notebook 下方会打印实际导入的包路径，应该位于**当前仓库**。iPad 只是浏览器界面，Python 运行在 Mac 或云端服务器上。需要更新 `src/` 中的函数时保存文件，再重新运行相关单元格。可使用 Jupyter 的 “Restart Kernel and Run All Cells” 检查是否依赖先前的隐藏状态。

In [ ]:
from pathlib import Path
import sys

# Find this checkout whether Jupyter starts at the repository root or notebooks/.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/three_link_robot').is_dir()), None)
if root is None:
    raise RuntimeError('Open this notebook inside the three_link_robot checkout.')
for location in (root, root / 'src'):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))
%load_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import three_link_robot
from examples.demo_osc import make_experiment, run_experiment, plot_experiment
from three_link_robot.kinematics import forward_kinematics, jacobian, jacobian_dot_times_dq
from three_link_robot.dynamics import mass_matrix, coriolis_forces, gravity_forces, forward_dynamics
from three_link_robot.operational_space import (task_diagnostics, check_task_jacobian,
                                                  task_inertia, task_torque, nullspace_projector,
                                                  TaskSingularityError)
from three_link_robot.visualization import plot_robot, animate_robot
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.grid': True, 'grid.alpha': .25})
print('Python:', sys.executable)
print('Package:', three_link_robot.__file__)
print('Project:', root)

## 2. 从关节状态走到末端位置

约定 $q,\dot q,\ddot q\in\mathbb{R}^3$；末端位置、速度、加速度属于 $\mathbb{R}^2$。`forward_kinematics(q, robot)` 返回基座与三个连杆端点的坐标，末端是最后一行。`jacobian(q, robot)` 的形状为 `(2, 3)`，满足 $\dot x=J(q)\dot q$。

先看默认参数。长度单位为 m，质量为 kg，角度为 rad；世界坐标系的重力方向是 $-y$。这一步不依赖任何 OSC 练习。

In [ ]:
TASK = 'hold'  # Later try 'circle' and 'nullspace'.
DURATION = 5.0  # s
SAMPLES = 151   # Number of saved output points; not the controller sampling rate.
robot, initial, trajectory, controller = make_experiment(TASK)
q_start, dq_start = initial[:3], initial[3:]
points_start = forward_kinematics(q_start, robot)
J_start = jacobian(q_start, robot)
diag = task_diagnostics(q_start, robot)
print('Link lengths [m]:', robot.lengths)
print('Masses [kg]:', robot.masses)
print('q [rad]:', q_start, '  dq [rad/s]:', dq_start)
print('End-effector xy [m]:', points_start[-1])
print('Shapes: points', points_start.shape, 'J', J_start.shape)
print('Jacobian rank:', diag['rank'])
print('Singular values [m/rad]:', diag['singular_values'])
print('Manipulability [(m/rad)^2]:', diag['manipulability'])

### 观察初始构型和目标

`hold` 的目标比初始末端位置高、远各 2 cm；`circle` 从初始末端位置开始画半径 5 cm 的圆；`nullspace` 保持末端原位但调整关节姿态。虚线是**参考轨迹**，此时还没有运行控制器。初始关节角是弯曲构型；全零伸直构型的二维位置 Jacobian 会失去一行独立方向。

In [ ]:
ref_times = np.linspace(0.0, DURATION, SAMPLES)
ref_xy = np.array([trajectory(t).x for t in ref_times])
ax, _ = plot_robot(points_start, reach=sum(robot.lengths))
ax.plot(ref_xy[:, 0], ref_xy[:, 1], '--x', markevery=[0], label='Target')
ax.set_title('Initial arm and desired path — no simulation yet')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.legend()
plt.show()
print('Reference at t=0:', trajectory(0.0))

### 先核对参考轨迹的导数

`TaskReference` 同时保存 $x_d,\dot x_d,\ddot x_d$。定点参考的速度和加速度应为零；圆轨迹则应连续变化。下面用中心差分检查位置的一阶导数、速度的一阶导数是否分别对应参考速度和加速度。这是对**参考生成器**的检查，与控制效果无关。

In [ ]:
t_check, dt_check = 0.7, 1e-5
r0 = trajectory(t_check)
r_plus, r_minus = trajectory(t_check + dt_check), trajectory(t_check - dt_check)
dx_fd = (r_plus.x - r_minus.x) / (2 * dt_check)
ddx_fd = (r_plus.dx - r_minus.dx) / (2 * dt_check)
print('dx reference [m/s]:', r0.dx, '  numerical:', dx_fd)
print('ddx reference [m/s²]:', r0.ddx, '  numerical:', ddx_fd)
print('Derivative errors:', np.linalg.norm(dx_fd-r0.dx), np.linalg.norm(ddx_fd-r0.ddx))

## 3. OSC-01：理解 $\dot J\dot q$

对 $\dot x=J(q)\dot q$ 再求导：

$$
\ddot x=J(q)\ddot q+\dot J(q,\dot q)\dot q.
$$

前项来自关节加速度，后项是关节正在运动时 Jacobian 改变造成的末端加速度。练习文件是 `src/three_link_robot/kinematics.py` 的 `jacobian_dot_times_dq(q, dq, robot)`；返回形状必须是 `(2,)`。可先考虑：若所有 $\dot q=0$，结果应是什么？它与 $J\ddot q$ 有何区别？

**自测方法**：沿 $\dot q$ 方向微扰构型，在两个位置计算 Jacobian，用中心差分近似 $\dot J\dot q$。下方数值值只作为核对依据，不会写入练习函数或被闭环控制器调用。

In [ ]:
q_probe = q_start.copy()
dq_probe = np.array([0.4, -0.3, 0.2])  # rad/s; nonzero to expose the velocity term.
eps = 1e-5
J_plus = jacobian(q_probe + eps*dq_probe, robot)
J_minus = jacobian(q_probe - eps*dq_probe, robot)
jdot_dq_fd = ((J_plus - J_minus) / (2*eps)) @ dq_probe
print('Numerical Jdot dq [m/s²]:', jdot_dq_fd)
jdot_dq_own = None
try:
    jdot_dq_own = jacobian_dot_times_dq(q_probe, dq_probe, robot)
except NotImplementedError as error:
    print('OSC-01 待完成：', error)
else:
    print('Your result [m/s²]:', jdot_dq_own)
    print('Difference norm:', np.linalg.norm(jdot_dq_own - jdot_dq_fd))

## 4. OSC-02：任务空间惯量

已有动力学满足 $M(q)\ddot q+c(q,\dot q)+g(q)=\tau$，其中 $M$ 是 `(3, 3)`；位置任务只有两个维度。定义

$$
\Lambda=(JM^{-1}J^T)^{-1},\qquad \Lambda\in\mathbb R^{2\times2}.
$$

先算 $X$，使 $MX=J^T$，再令 $A=JX$；可以用 `np.linalg.solve`，无须显式求 $M^{-1}$。实现 `task_inertia(J, M)` 后，核对 $\Lambda=\Lambda^T$、特征值均为正、$A\Lambda\approx I_2$。这里的前提是 $M$ 正定、$J$ 满行秩；`check_task_jacobian` 对当前构型给出显式保护，但不保证整条路径安全。

In [ ]:
M_start = mass_matrix(q_probe, robot)
J_probe = jacobian(q_probe, robot)
check_task_jacobian(J_probe)
X = np.linalg.solve(M_start, J_probe.T)
A = J_probe @ X
print('M shape:', M_start.shape, '  symmetry error:', np.linalg.norm(M_start-M_start.T))
print('M eigenvalues:', np.linalg.eigvalsh(M_start))
print('A shape:', A.shape, '  A eigenvalues:', np.linalg.eigvalsh(A))
try:
    Lambda = task_inertia(J_probe, M_start)
except NotImplementedError as error:
    Lambda = None
    print('OSC-02 待完成：', error)
else:
    print('Lambda shape:', np.shape(Lambda))
    print('Symmetry error:', np.linalg.norm(Lambda-Lambda.T))
    print('Lambda eigenvalues:', np.linalg.eigvalsh(Lambda))
    print('Inverse relation error:', np.linalg.norm(A @ Lambda - np.eye(2)))

## 5. OSC-03：末端加速度到关节力矩

控制器先将任务误差变成期望末端加速度：

$$
a_{cmd}=\ddot x_d+K_d(\dot x_d-J\dot q)+K_p(x_d-x).
$$

本工程使用**完整关节偏置补偿** $h=c+g$：

$$
\tau=h+J^T\Lambda(a_{cmd}-\dot J\dot q).
$$

`task_torque` 返回 `(3,)`，单位为 N·m。注意 $h$ 是三维关节力矩，$\Lambda(a_{cmd}-\dot J\dot q)$ 是二维任务力；先乘 $J^T$ 才能相加。外层 `osc_control` 已计算反馈和模型项，因此只需实现 `src/three_link_robot/operational_space.py` 中的力矩组装函数。检查时，把得到的 $\tau$ 代入已有正向动力学，验证实际 $J\ddot q+\dot J\dot q$ 接近 $a_{cmd}$；仓库测试会完成这一检查。

In [ ]:
# Inspect the model bias separately; this does not call unfinished OSC functions.
h_probe = coriolis_forces(q_probe, dq_probe, robot) + gravity_forces(q_probe, robot)
r_probe = trajectory(0.0)
x_probe = forward_kinematics(q_probe, robot)[-1]
dx_probe = J_probe @ dq_probe
kp_task = np.array([25.0, 25.0])  # 1/s², same defaults as osc_control
kd_task = np.array([10.0, 10.0])  # 1/s
position_error = r_probe.x - x_probe
velocity_error = r_probe.dx - dx_probe
a_cmd = r_probe.ddx + kd_task*velocity_error + kp_task*position_error
print('Position error [m]:', position_error)
print('Velocity error [m/s]:', velocity_error)
print('Desired task acceleration [m/s²]:', a_cmd)
print('h = c + g [N·m]:', h_probe)
if Lambda is None or jdot_dq_own is None:
    print('完成 OSC-01 和 OSC-02 后，可在此检查 OSC-03 的实际加速度。')
else:
    try:
        tau_probe = task_torque(J_probe, Lambda, h_probe, a_cmd, jdot_dq_own)
    except NotImplementedError as error:
        print('OSC-03 待完成：', error)
    else:
        ddq_actual = forward_dynamics(q_probe, dq_probe, tau_probe, robot)
        a_actual = J_probe @ ddq_actual + jdot_dq_fd  # Independent numerical bias.
        print('Joint torques [N·m]:', tau_probe)
        print('Actual task acceleration [m/s²]:', a_actual)
        print('Acceleration difference norm [m/s²]:', np.linalg.norm(a_actual-a_cmd))

## 6. OSC-04：多出的关节自由度

三关节控制二维位置，正常情况下还有一维零空间。定义动力学一致广义逆和速度投影：

$$
\bar J=M^{-1}J^T\Lambda,\qquad N=I_3-\bar JJ.
$$

$N$ 通常**不对称**。若想施加关节姿态力矩 $\tau_0$，控制律使用 $N^T\tau_0$，而不是 $N\tau_0$：

$$
\tau=h+J^T\Lambda(a_{cmd}-\dot J\dot q)+N^T\tau_0.
$$

实现 `nullspace_projector(J, M, Lambda)` 后，检查 $JN\approx0$、$N^2\approx N$、$JM^{-1}N^T\approx0$。`nullspace` 实验只尝试移动姿态目标，**不能保证三个关节都到任意指定角度**；优先检查末端位置是否仍保持。

In [ ]:
if Lambda is None:
    print('先完成 OSC-02，再检查零空间投影。')
else:
    try:
        N = nullspace_projector(J_probe, M_start, Lambda)
    except NotImplementedError as error:
        print('OSC-04 待完成：', error)
    else:
        print('N shape:', np.shape(N), '  rank:', np.linalg.matrix_rank(N, tol=1e-9))
        print('||J N||:', np.linalg.norm(J_probe @ N))
        print('||N²-N||:', np.linalg.norm(N @ N - N))
        print('||J M^-1 Nᵀ||:', np.linalg.norm(J_probe @ np.linalg.solve(M_start, N.T)))

## 7. 运行闭环实验

先用 `TASK='hold'`。完成 OSC-01～03 后，控制器应对一个小位置阶跃作出响应；然后改为 `circle` 看连续跟踪。`nullspace` 还要求 OSC-04。修改前面的 `TASK` 后，**从参数单元格重新运行到这里**，避免旧参考和新实验混用。

流程是：`trajectory(t)` 给出目标 → `osc_control` 计算关节力矩 → `forward_dynamics` 求实际关节加速度 → 积分器得到新的 $q,\dot q$。这里的 RK45 是连续反馈仿真；`SAMPLES` 是保存结果的采样点数，不代表控制器每隔固定时间执行一次。

下方先调用一次控制器。未填写公式或遇到奇异构型时只打印原因；其他错误会显示完整异常，便于你定位代码问题。

In [ ]:
data = None  # Avoid showing a stale result after an unsuccessful rerun.
try:
    controller(0.0, q_start, dq_start, trajectory(0.0), robot)
except NotImplementedError as error:
    print('尚未运行闭环：', error)
    print('填写相应的 OSC 函数后，从参数单元格重新运行。')
except TaskSingularityError as error:
    print('构型检查未通过：', error)
else:
    data = run_experiment(TASK, duration=DURATION, samples=SAMPLES)
    print(f"Final position error: {data['error'][-1]:.6g} m")
    fig = plot_experiment(data)
    plt.show()

### 怎样读四张图

- **左上**：末端实际路径与参考路径。定点实验的参考是一个点；圆轨迹实验从初始点开始。
- **右上**：位置误差的范数，单位 m。定点实验看是否逐渐下降；圆轨迹看跟踪误差是否保持有界且较小。
- **左下**：三个关节角随时间变化。零空间实验可在这里看到姿态变化；不能只凭这一图判断末端是否保持。
- **右下**：二维 Jacobian 的最小奇异值。若接近保护阈值 `1e-4 m/rad`，未阻尼的 OSC 可能无法继续使用。

误差曲线并不是稳定性证明；这里控制器与被控对象使用同一模型，且没有力矩饱和、噪声和离散采样。下面的数值摘要只在真实仿真成功后生成。

In [ ]:
if data is None:
    print('还没有闭环结果；先完成缺失公式。')
else:
    states = data['result'].states
    print(f"Initial / final error [m]: {data['error'][0]:.6g} / {data['error'][-1]:.6g}")
    print(f"RMS error [m]: {np.sqrt(np.mean(data['error']**2)):.6g}")
    print(f"Minimum sigma over saved path [m/rad]: {data['sigma_min'].min():.6g}")
    print('Joint displacement [rad]:', states[-1, :3] - states[0, :3])
    if TASK == 'nullspace':
        print('Check that the end-effector remains near its initial position while q changes.')

### 可选：在浏览器中播放动画

动画只使用已经求出的闭环状态；不会额外计算或修复控制器。默认关闭，避免第一次打开 Notebook 时生成较大的 HTML。运行成功后将 `SHOW_ANIMATION` 改为 `True`，再执行此单元格。

In [ ]:
SHOW_ANIMATION = False
if data is not None and SHOW_ANIMATION:
    time_step_ms = 1000 * (data['result'].t[1] - data['result'].t[0])
    figure, animation = animate_robot(data['points'], reach=sum(robot.lengths),
                                       interval_ms=time_step_ms)
    figure.axes[0].set_title('OSC closed-loop dynamics')
    display(HTML(animation.to_jshtml()))
    plt.close(figure)
elif data is None:
    print('没有闭环数据；不生成动画。')
else:
    print('动画已关闭；设置 SHOW_ANIMATION=True 后重新运行。')

## 8. 自测清单与下一步

在工程根目录的终端执行 `python -m pytest tests/test_osc.py -q -rs`。测试显示 **skipped 是未完成**，不是验证通过。建议每写完一个函数就运行一次，而不是四个函数一起写完再排错。

| 阶段 | 函数 | 通过时应看到什么 |
| --- | --- | --- |
| OSC-01 | `jacobian_dot_times_dq` | 与沿 $\dot q$ 方向的 Jacobian 中心差分一致；$\dot q=0$ 时结果为零 |
| OSC-02 | `task_inertia` | $\Lambda$ 对称正定，$JM^{-1}J^T\Lambda\approx I_2$ |
| OSC-03 | `task_torque` | 正向动力学算出的末端加速度接近 $a_{cmd}$；`hold` 和 `circle` 能运行 |
| OSC-04 | `nullspace_projector` | $JN\approx0$、$JM^{-1}N^T\approx0$；`nullspace` 能运行且末端仍受控 |

若误差大，先检查形状 `(2,)` / `(3,)`、单位、矩阵乘法顺序和 $N^T$ 的转置。若轨迹中触发奇异构型，先查看最小奇异值曲线和参考路径，不要把保护异常直接删除。完成这些理想模型练习后，再研究参数失配、力矩饱和、离散控制周期和阻尼逆的取舍。